# Welch's $t$-Test Calculation Notebook

This notebook demonstrates the step-by-step mathematical formulation and Python execution of **Welch's unequal variances $t$-test** on the 350-run physical trial dataset (`artifacts/dataset1_ota_performance.csv` and `artifacts/dataset4_e2e_diagnostic.csv`).

---

## Mathematical Formulation

### 1. Welch's $t$-Statistic
Unlike Student's $t$-test, Welch's $t$-test does not assume equal variances between two samples $X_1$ and $X_2$:

$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

where:
- $\bar{X}_1, \bar{X}_2$ are the sample means.
- $s_1^2, s_2^2$ are the sample variances.
- $n_1, n_2$ are the sample sizes.

### 2. Welch–Satterthwaite Degrees of Freedom ($\nu$)
$$\nu \approx \frac{\left(\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right)^2}{\frac{\left(\frac{s_1^2}{n_1}\right)^2}{n_1 - 1} + \frac{\left(\frac{s_2^2}{n_2}\right)^2}{n_2 - 1}}$$

### 3. $p$-Value Determination
$$p = 2 \cdot \left(1 - F(|t|; \nu)\right)$$
where $F(t; \nu)$ is the cumulative distribution function (CDF) of Student's $t$-distribution with $\nu$ degrees of freedom.

In [1]:
import math
import os

import pandas as pd

# Load real physical trial telemetry datasets
df1 = pd.read_csv("artifacts/dataset1_ota_performance.csv")
print("Datasets loaded successfully.")
print(f"Dataset 1 Total Rows: {len(df1)}")
if os.path.exists("artifacts/dataset4_e2e_diagnostic.csv"):
    df4 = pd.read_csv("artifacts/dataset4_e2e_diagnostic.csv")
    print(f"Dataset 4 Total Rows: {len(df4)}")

Datasets loaded successfully.
Dataset 1 Total Rows: 350
Dataset 4 Total Rows: 350


## Step-by-Step Welch's $t$-Test Function

We define a custom Python function to compute the mean, variance, standard error, degrees of freedom (Welch-Satterthwaite), and $t$-statistic manually.

In [2]:
def betainc_cf(a, b, x):
    MAXIT = 200
    EPS = 3.0e-15
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 - qab * x / qap
    if abs(d) < EPS:
        d = EPS
    d = 1.0 / d
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < EPS:
            d = EPS
        c = 1.0 + aa / c
        if abs(c) < EPS:
            c = EPS
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < EPS:
            d = EPS
        c = 1.0 + aa / c
        if abs(c) < EPS:
            c = EPS
        d = 1.0 / d
        del_h = d * c
        h *= del_h
        if abs(del_h - 1.0) < EPS:
            break
    return h


def betainc(a, b, x):
    if x <= 0.0:
        return 0.0
    if x >= 1.0:
        return 1.0
    log_bt = (
        math.lgamma(a + b)
        - math.lgamma(a)
        - math.lgamma(b)
        + a * math.log(x)
        + b * math.log(1.0 - x)
    )
    bt = math.exp(log_bt)
    if x < (a + 1.0) / (a + b + 2.0):
        return bt * betainc_cf(a, b, x) / a
    else:
        return 1.0 - bt * betainc_cf(b, a, 1.0 - x) / b


def student_t_pvalue(t, df):
    t_abs = abs(t)
    x = df / (df + t_abs**2)
    return betainc(df / 2.0, 0.5, x)


def calculate_welch_ttest(sample1, sample2, metric_name="Metric"):
    n1, n2 = len(sample1), len(sample2)
    m1, m2 = sum(sample1) / n1, sum(sample2) / n2
    v1 = sum((x - m1) ** 2 for x in sample1) / (n1 - 1)
    v2 = sum((x - m2) ** 2 for x in sample2) / (n2 - 1)

    se = math.sqrt(v1 / n1 + v2 / n2)
    t_stat = (m1 - m2) / se
    df_denom = ((v1 / n1) ** 2 / (n1 - 1)) + ((v2 / n2) ** 2 / (n2 - 1))
    df = ((v1 / n1 + v2 / n2) ** 2) / df_denom

    # Exact Student's t-distribution two-tailed p-value
    p_val = student_t_pvalue(t_stat, df)

    print(f"=== Welch's t-Test: {metric_name} ===")
    print(f"Sample 1 (aarch64, n={n1}): Mean = {m1:.6f}, Var = {v1:.6f}")
    print(f"Sample 2 (x86_64, n={n2}): Mean = {m2:.6f}, Var = {v2:.6f}")
    print(f"Mean Difference (aarch64 - x86_64): {m1 - m2:.6f}")
    print(f"Welch t-statistic: {t_stat:.4f}")
    print(f"Degrees of Freedom (df): {df:.2f}")
    if p_val < 0.0001:
        print(f"p-value: {p_val:.2e} (p < 0.001, Overwhelmingly Significant)")
    else:
        print(f"p-value: {p_val:.6f} (p > 0.05, Statistically Insignificant)")
    print(f"Statistically Significant (alpha=0.05): {p_val < 0.05}")
    return t_stat, df, p_val

## 1. CI Build Time Welch's $t$-Test (`x86_64` vs `aarch64`)

In [3]:
ci_x86 = list(df1[df1["Target Architecture"] == "x86_64"]["CI Build Time (sec)"])
ci_aarch = list(df1[df1["Target Architecture"] == "aarch64"]["CI Build Time (sec)"])
res_ci = calculate_welch_ttest(ci_aarch, ci_x86, "CI Build Time (aarch64 vs x86_64)")

=== Welch's t-Test: CI Build Time (aarch64 vs x86_64) ===
Sample 1 (aarch64, n=175): Mean = 42.521943, Var = 1.414290
Sample 2 (x86_64, n=175): Mean = 38.675714, Var = 0.902448
Mean Difference (aarch64 - x86_64): 3.846229
Welch t-statistic: 33.4284
Degrees of Freedom (df): 331.80
p-value: 2.99e-108 (p < 0.001, Overwhelmingly Significant)
Statistically Significant (alpha=0.05): True
